# 🎯 M2 Angle Regression Training (Improved)

Train model để dự đoán góc xoay của đồng hồ nước (-180° đến 180°)

## ✨ Best Practices Applied:
1. **Cos/Sin Representation**: Output vector thay vì góc trực tiếp (tránh discontinuity)
2. **Cosine Similarity Loss**: Loss tối ưu cho angle regression
3. **Continuous Angle**: Vector luôn mượt mà trên toàn vòng tròn

## 📊 Dataset
- **Train**: 9,996 ảnh synthetic
- **Val**: 3,000 ảnh synthetic
- **Range**: -180° đến 180° (full 360°)

## 🚀 Quick Start
1. Chạy từng cell theo thứ tự
2. File ZIP tự động tìm trong Google Drive

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')
print("\n✅ Google Drive mounted successfully!")

📂 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted successfully!


## 2️⃣ Tìm & Extract Dataset từ Google Drive

In [ ]:
# Tìm file dataset trong Google Drive
import os

# Đường dẫn có thể có
POSSIBLE_PATHS = [
    '/content/drive/MyDrive/WaterMeter_Project/m2_angle_dataset.zip',
    '/content/drive/MyDrive/WaterMeter_Project/data/m2_angle_dataset.zip',
    '/content/drive/MyDrive/m2_angle_dataset.zip',
]

print("🔍 Tìm file dataset trong Google Drive...\n")

# Tìm file ZIP
ZIP_PATH = None
for path in POSSIBLE_PATHS:
    if os.path.exists(path):
        ZIP_PATH = path
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"✅ Tìm thấy file: {path}")
        print(f"   Size: {size_mb:.1f} MB")
        break

# Nếu không tìm thấy, tìm trong thư mục watermeter_Project
if ZIP_PATH is None:
    print("\n🔍 Tìm trong thư mục watermeter_Project...")
    PROJECT_DIR = '/content/drive/MyDrive/watermeter_Project'
    if os.path.exists(PROJECT_DIR):
        for root, dirs, files in os.walk(PROJECT_DIR):
            for file in files:
                if file.endswith('.zip') and 'm2_angle' in file.lower():
                    ZIP_PATH = os.path.join(root, file)
                    size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
                    print(f"✅ Tìm thấy: {file}")
                    print(f"   Path: {ZIP_PATH}")
                    print(f"   Size: {size_mb:.1f} MB")
                    break
            if ZIP_PATH:
                break

if ZIP_PATH is None:
    print("\n❌ Không tìm thấy file dataset!")
    print("\n📝 Vui lòng đảm bảo file ZIP được đặt tại một trong các đường dẫn:")
    for path in POSSIBLE_PATHS:
        print(f"   - {path}")
    print("\nHoặc upload file m2_angle_dataset.zip vào Google Drive.")
else:
    print(f"\n✅ Sẽ sử dụng file: {ZIP_PATH}")
    DATASET_ZIP = ZIP_PATH

🔍 Tìm file dataset trong Google Drive...

✅ Tìm thấy file: /content/drive/MyDrive/WaterMeter_Project/m2_angle_dataset.zip
   Size: 2358.8 MB

✅ Sẽ sử dụng file: /content/drive/MyDrive/WaterMeter_Project/m2_angle_dataset.zip


In [ ]:
# Giải nén dataset
import zipfile

DATASET_ZIP = DATASET_ZIP  # Từ cell trước
EXTRACT_PATH = '/content/m2_angle_dataset'

print(f"\n📦 Giải nén dataset...")
print(f"   Source: {DATASET_ZIP}")
print(f"   Target: {EXTRACT_PATH}\n")

# Extract
with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("✅ Giải nén hoànất!\n")

# Kiểm tra structure
print("📁 Dataset structure:")
for root, dirs, files in os.walk(EXTRACT_PATH):
    level = root.replace(EXTRACT_PATH, '').count(os.sep)
    if level <= 2:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subdirs = [d for d in dirs if not d.startswith('.')]
        for subdir in subdirs:
            subpath = os.path.join(root, subdir)
            if os.path.isdir(subpath):
                count = len([f for f in os.listdir(subpath) if f.endswith(('.jpg', '.png'))])
                print(f"{indent}  {subdir}/ ({count} images)")

# Đếm số lượng
train_count = len(os.listdir(f"{EXTRACT_PATH}/train/images"))
val_count = len(os.listdir(f"{EXTRACT_PATH}/val/images"))
print(f"\n📊 Dataset info:")
print(f"   Train: {train_count:,} images")
print(f"   Val: {val_count:,} images")
print(f"   Total: {train_count + val_count:,} images")

DATASET_PATH = EXTRACT_PATH


📦 Giải nén dataset...
   Source: /content/drive/MyDrive/WaterMeter_Project/m2_angle_dataset.zip
   Target: /content/m2_angle_dataset

✅ Giải nén hoànất!

📁 Dataset structure:
m2_angle_dataset/
  train/ (0 images)
  val/ (0 images)
  train/
    images/ (9996 images)
    images/
  val/
    images/ (3000 images)
    images/

📊 Dataset info:
   Train: 9,996 images
   Val: 3,000 images
   Total: 12,996 images


## 3️⃣ Setup Environment

In [ ]:
# Cài đặt thư viện
print("📚 Cài đặt thư viện...")
!pip install torch torchvision tqdm pandas scikit-learn -q
print("✅ Cài đặt hoàn tất!\n")

# Import thư viện
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print(f"   ⚠️ Không có GPU - training sẽ rất chậm!")

📚 Cài đặt thư viện...
✅ Cài đặt hoàn tất!

💻 Device: cuda
   GPU: Tesla T4
   Memory: 14.6 GB


## 4️⃣ Define Model & Dataset

In [ ]:
class AngleRegressionModel(nn.Module):
    """
    Improved Angle Regression Model using Cos/Sin representation
    Output: Vector (cos, sin) normalized to unit length
    """
    def __init__(self, pretrained=True):
        super().__init__()
        # Backbone: ResNet18
        resnet = models.resnet18(weights='DEFAULT' if pretrained else None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # [B,512,7,7]

        # Angle regression head (output: cos, sin)
        self.angle_head = nn.Sequential(
            nn.Flatten(),  # 512*7*7 = 25088
            nn.Linear(512*7*7, 1024),
            nn.GroupNorm(32, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.GroupNorm(16, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2),  # Output: (cos, sin)
            nn.Tanh()  # Constrain to [-1, 1]
        )

    def forward(self, x):
        feats = self.backbone(x)  # [B,512,7,7]
        vec = self.angle_head(feats)  # [B,2] in [-1, 1]
        # Normalize to unit vector
        vec_normalized = nn.functional.normalize(vec, p=2, dim=1)
        return vec_normalized  # [B,2]


class AngleDataset(Dataset):
    """Dataset cho angle regression với cos/sin labels"""
    def __init__(self, img_dir, labels_csv, transform=None):
        self.df = pd.read_csv(labels_csv)
        self.img_dir = img_dir
        self.transform = transform

        # Lọc ảnh hợp lệ
        self.df['exists'] = self.df['filename'].apply(
            lambda f: os.path.exists(os.path.join(self.img_dir, f))
        )
        self.df = self.df[self.df['exists'] == True].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        img = Image.open(img_path).convert('RGB')

        # Convert angle to cos/sin vector (continuous representation)
        angle_rad = np.radians(row['angle'])
        cos_val = np.cos(angle_rad)
        sin_val = np.sin(angle_rad)

        if self.transform:
            img = self.transform(img)

        # Return image with (cos, sin) label
        label = torch.tensor([cos_val, sin_val], dtype=torch.float32)
        return img, label


def angle_from_vector(cos_sin):
    """Convert (cos, sin) vector back to angle in degrees"""
    cos_val, sin_val = cos_sin
    angle_rad = np.arctan2(sin_val, cos_val)
    angle_deg = np.degrees(angle_rad)
    return angle_deg


print("✅ Model & Dataset đã được định nghĩa!")
print("📌 Using Cos/Sin representation for continuous angle prediction")

✅ Model & Dataset đã được định nghĩa!
📌 Using Cos/Sin representation for continuous angle prediction


In [ ]:
# Paths
TRAIN_IMG_DIR = f"{DATASET_PATH}/train/images"
VAL_IMG_DIR = f"{DATASET_PATH}/val/images"
TRAIN_LABELS = f"{DATASET_PATH}/train_labels.csv"
VAL_LABELS = f"{DATASET_PATH}/val_labels.csv"

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),  # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets
print("📊 Tạo datasets...")
train_dataset = AngleDataset(TRAIN_IMG_DIR, TRAIN_LABELS, train_transform)
val_dataset = AngleDataset(VAL_IMG_DIR, VAL_LABELS, val_transform)
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val: {len(val_dataset):,} samples")

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

print("\n✅ Datasets sẵn sàng!")

📊 Tạo datasets...
   Train: 9,996 samples
   Val: 3,000 samples
   Batch size: 32
   Train batches: 313
   Val batches: 94

✅ Datasets sẵn sàng!


## 5️⃣ Training Configuration

In [ ]:
# Training config
EPOCHS = 20
PATIENCE = 25  # Early stopping
LEARNING_RATE = 1e-4
BEST_MODEL_PATH = '/content/m2_angle_model_best.pth'
GDRIVE_DIR = '/content/drive/MyDrive/watermeter_Project'
LAST_MODEL_PATH = f'{GDRIVE_DIR}/m2_angle_model_last.pth'

# Create Google Drive directory if not exists
import os
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f"📁 Google Drive directory: {GDRIVE_DIR}")

# Create model
print("🧠 Tạo model...")
model = AngleRegressionModel(pretrained=True).to(device)
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Cosine Similarity Loss (Best for angle regression!)
class CosineAngleLoss(nn.Module):
    """Cosine Similarity Loss cho vector angle prediction"""
    def __init__(self):
        super().__init__()
        self.cosine_sim = nn.CosineSimilarity(dim=1)

    def forward(self, pred, target):
        # pred: [B, 2] (cos, sin) normalized
        # target: [B, 2] (cos, sin)
        # Loss = 1 - cosine_similarity (range: 0 to 2)
        return torch.mean(1 - self.cosine_sim(pred, target))

criterion = CosineAngleLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-7
)

print(f"\n⚙️ Training Configuration:")
print(f"   Epochs: {EPOCHS}")
print(f"   Patience: {PATIENCE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Loss: Cosine Similarity Loss (vector-based)")
print(f"   Best model: {BEST_MODEL_PATH}")
print(f"   Last model: {LAST_MODEL_PATH}")

print("\n✅ Ready to train! 🚀")

📁 Google Drive directory: /content/drive/MyDrive/watermeter_Project
🧠 Tạo model...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


   Parameters: 37,396,546
   Trainable: 37,396,546

⚙️ Training Configuration:
   Epochs: 20
   Patience: 25
   Learning rate: 0.0001
   Loss: Cosine Similarity Loss (vector-based)
   Best model: /content/m2_angle_model_best.pth
   Last model: /content/drive/MyDrive/watermeter_Project/m2_angle_model_last.pth

✅ Ready to train! 🚀


## 6️⃣ Train Model (⚠️ Chạy qua đêm ~1 tiếng)

In [ ]:
# Training loop
best_val_loss = float('inf')
patience_counter = 0

print("\n" + "="*60)
print("🚀 BẮT ĐẦU TRAINING")
print("="*60 + "\n")
print("⏰ Thời gian dự kiến: ~1 tiếng cho 20 epochs")
print("💡 Tip: Chạy qua đêm để Colab free không bị timeout")
print("\n")

start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:3d}/{EPOCHS}", unit="batch")
    for images, angles in pbar:
        images = images.to(device)
        angles = angles.to(device)

        optimizer.zero_grad()
        pred_angles = model(images)
        loss = criterion(pred_angles, angles)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_loss /= len(train_dataset)

    # ========== VALIDATION ==========
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, angles in val_loader:
            images = images.to(device)
            angles = angles.to(device)
            pred_angles = model(images)
            loss = criterion(pred_angles, angles)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_dataset)

    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    # Print progress
    elapsed = (time.time() - start_time) / 60
    print(f"E{epoch+1:3d}/{EPOCHS} | T:{train_loss:.5f} V:{val_loss:.5f} | LR:{current_lr:.2e} | {epoch_time:.0f}s | ⏱️{elapsed:.0f}m")

    # ========== SAVE BEST MODEL ==========
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        # Save best model locally
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, BEST_MODEL_PATH)

        # Copy to Google Drive (with directory check)
        try:
            import shutil
            gdrive_save_path = f"{LAST_MODEL_PATH}_epoch{epoch+1}.pth"
            shutil.copy(BEST_MODEL_PATH, gdrive_save_path)
            print(f"   ✓ Best model saved! (V:{val_loss:.5f})")
        except Exception as e:
            print(f"   ⚠️ Local saved only! Google Drive error: {e}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}!")
            break

total_time = time.time() - start_time

print("\n" + "="*60)
print("✅ TRAINING HOÀN TẤT!")
print("="*60)
print(f"⏱️ Tổng thời gian: {total_time/60:.1f} phút")
print(f"📉 Best Val Loss: {best_val_loss:.5f}")
print(f"💾 Model saved to: {BEST_MODEL_PATH}")
print(f"💾 Google Drive backup: {GDRIVE_DIR}/")


🚀 BẮT ĐẦU TRAINING

⏰ Thời gian dự kiến: ~1 tiếng cho 20 epochs
💡 Tip: Chạy qua đêm để Colab free không bị timeout




Epoch   1/20: 100%|██████████| 313/313 [02:25<00:00,  2.15batch/s, loss=0.0127]


E  1/20 | T:0.04564 V:0.00584 | LR:9.94e-05 | 169s | ⏱️3m
   ✓ Best model saved! (V:0.00584)


Epoch   2/20: 100%|██████████| 313/313 [02:23<00:00,  2.18batch/s, loss=0.0068]


E  2/20 | T:0.00594 V:0.00283 | LR:9.76e-05 | 167s | ⏱️6m
   ✓ Best model saved! (V:0.00283)


...

E 18/20 | T:0.00043 V:0.00056 | LR:2.54e-06 | 155s | ⏱️48m
   ✓ Best model saved! (V:0.00056)


Epoch  19/20: 100%|██████████| 313/313 [02:11<00:00,  2.38batch/s, loss=0.0005]


E 19/20 | T:0.00042 V:0.00057 | LR:7.15e-07 | 154s | ⏱️51m


Epoch  20/20: 100%|██████████| 313/313 [02:08<00:00,  2.43batch/s, loss=0.0006]


E 20/20 | T:0.00041 V:0.00059 | LR:1.00e-04 | 153s | ⏱️54m

✅ TRAINING HOÀN TẤT!
⏱️ Tổng thời gian: 53.7 phút
📉 Best Val Loss: 0.00056
💾 Model saved to: /content/m2_angle_model_best.pth
💾 Google Drive backup: /content/drive/MyDrive/watermeter_Project/


## 7️⃣ Download Model

In [ ]:
# Download best model về máy tính
from google.colab import files

print("📥 Downloading trained model...\n")
files.download(BEST_MODEL_PATH)

print("\n✅ Model đã được download về máy tính!")
print("\n📝 File: m2_angle_model_best.pth")
print("📁 Lưu vào: Downloads folder")

print("\n💡 Model cũng đã được lưu trong Google Drive tại:")
print(f"   {GDRIVE_DIR}/m2_angle_model_last.pth_epoch<X>.pth")

📥 Downloading trained model...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Model đã được download về máy tính!

📝 File: m2_angle_model_best.pth
📁 Lưu vào: Downloads folder

💡 Model cũng đã được lưu trong Google Drive tại:
   /content/drive/MyDrive/watermeter_Project/m2_angle_model_last.pth_epoch<X>.pth


## 8️⃣ Evaluate Model

In [ ]:
# Load best model
checkpoint = torch.load(BEST_MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Test trên validation set
print("\n🧪 Evaluating model on validation set...\n")

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        pred_vecs = model(images)  # [B, 2] (cos, sin)

        # Convert vectors back to angles
        for pred_vec, label in zip(pred_vecs.cpu(), labels):
            # Prediction: vector → angle
            cos_val, sin_val = pred_vec.numpy()
            pred_angle = np.degrees(np.arctan2(sin_val, cos_val))

            # Label: vector → angle
            cos_lbl, sin_lbl = label.numpy()
            label_angle = np.degrees(np.arctan2(sin_lbl, cos_lbl))

            # Normalize to [-180, 180]
            if pred_angle > 180:
                pred_angle -= 360
            if label_angle > 180:
                label_angle -= 360

            all_preds.append(pred_angle)
            all_labels.append(label_angle)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
mae = np.mean(np.abs(all_preds - all_labels))
rmse = np.sqrt(np.mean((all_preds - all_labels)**2))

print(f"📊 Evaluation Results:")
print(f"   Mean Absolute Error: {mae:.2f}°")
print(f"   RMSE: {rmse:.2f}°")

# Accuracy within thresholds
print(f"\n📈 Accuracy within thresholds:")
for threshold in [1, 3, 5, 10]:
    acc = np.mean(np.abs(all_preds - all_labels) < threshold) * 100
    print(f"   Within ±{threshold:2d}°: {acc:5.2f}%")

# Cosine similarity score
from scipy.spatial.distance import cdist
print(f"\n📐 Vector Cosine Similarity: {(1 - mae/180)*100:.1f}%")

print("\n✅ Evaluation completed!")


🧪 Evaluating model on validation set...

📊 Evaluation Results:
   Mean Absolute Error: 5.50°
   RMSE: 38.26°

📈 Accuracy within thresholds:
   Within ± 1°: 43.83%
   Within ± 3°: 89.23%
   Within ± 5°: 97.03%
   Within ±10°: 98.73%

📐 Vector Cosine Similarity: 96.9%

✅ Evaluation completed!


## 📝 Notes & Tips

### ✨ Cải tiến mới (Best Practices Applied):

1. **🎯 Cos/Sin Representation** ✅
   - Output vector (cos, sin) thay vì góc trực tiếp
   - Tránh discontinuity tại 0°/360°
   - Suất mượt mà trên toàn vòng tròn

2. **📐 Cosine Similarity Loss** ✅
   - Loss tối ưu cho angle regression
   - Không bị nhảy loạn xạ như MSE Loss
   - Convergence nhanh hơn

3. **🔧 Continuous Angle Prediction** ✅
   - Vector luôn liên tục
   - Dùng arctan2(sin, cos) để lấy góc
   - Không bị vụn về nhãn

### 📍 Đường dẫn file ZIP
Đã được cấu hình để tự động tìm tại:
- `/content/drive/MyDrive/watermeter_Project/m2_angle_dataset.zip`
- `/content/drive/MyDrive/watermeter_Project/data/m2_angle_dataset.zip`

### ⏱️ Training Time
- **Colab T4 (Free)**: ~8 tiếng
- **Colab Pro (V100/A100)**: ~3.5 tiếng

### 💡 Tips
1. **Chạy qua đêm**: Bắt đầu trước khi đi ngủ
2. **Auto-save**: Model tự save vào Google Drive mỗi epoch
3. **Early stopping**: Tự động stop nếu không improve sau 25 epochs
4. **Reconnect**: Nếu mất kết nối → reconnect → chạy tiếp từ cell 6

### 📂 Files Output
- **Local**: `/content/m2_angle_model_best.pth`
- **Google Drive**: `/content/drive/MyDrive/watermeter_Project/m2_angle_model_last.pth_epoch<X>.pth`

### 🔧 Usage (Local)
```python
import torch
import numpy as np
from src.m2_orientation.model import AngleRegressionModel

# Load model
model = AngleRegressionModel()
checkpoint = torch.load('m2_angle_model_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Predict angle
def predict_angle(image, model):
    with torch.no_grad():
        vec = model(image)  # Returns (cos, sin)
        cos_val, sin_val = vec[0].cpu().numpy()
        angle = np.degrees(np.arctan2(sin_val, cos_val))
        return angle

angle = predict_angle('water_meter.jpg', model)
print(f'Angle: {angle:.2f}°')
```

### 📊 Model Architecture
```
ResNet18 Backbone (pre-trained)
    ↓
Flatten (512*7*7 = 25088)
    ↓
FC 1024 → GroupNorm → ReLU → Dropout(0.3)
    ↓
FC 512 → GroupNorm → ReLU → Dropout(0.2)
    ↓
FC 2 → Tanh → Normalize
    ↓
Output: (cos, sin) unit vector 🎯
```